In [14]:
import importlib
import os
import pickle
import random

import cv2
import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.optim import AdamW
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as func
from tqdm import tqdm

from src import model, modules, utils

importlib.reload(model)
importlib.reload(modules)
importlib.reload(utils)

from src.utils import MammoCNNDataset

In [15]:
SEED: int = 42
BATCH_SIZE: int = 32
NUM_WORKERS: int = 0
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE: list[int] = [416, 320]


# Set seed to ensure consistency across runs
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


g = torch.Generator()
g.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

#### Split the dataset by using 600 patients for a hold out test set. The remaining 4,400 patients are randomly partitioned into a MAMMO CNN training set, a Classifier training set, a Triage training set, and a validation set of 60%, 15%, 15%, and 10%, respectively.

In [16]:
metadata_path = os.path.join("vindr-mammo", "breast_metadata.parquet")
metadata_df = pd.read_parquet(metadata_path)

# Create a patient level dataframe
patient_df = (
    metadata_df[["study_id", "patient_cancer", "Patient's Age"]].drop_duplicates().reset_index(drop=True)
)

# Hold out test set 12%
train_val, test = train_test_split(
    patient_df,
    test_size=0.12,
    random_state=SEED,
    stratify=patient_df["patient_cancer"],
)

# Of the remaining 88%, split: 60% | 15% | 15% | 10%
# Sequential fractions:
#   val:             10/100 = 0.10  of remaining
#   triage_train:    15/90  ≈ 0.1667 of what's left after val
#   classifier_train:15/75  = 0.20  of what's left after val + triage
#   cnn_train:       60/60  = 1.0   (everything left)

# Split the remaining 4,400 patients
remaining_1, val = train_test_split(
    train_val,
    test_size=0.10,
    random_state=SEED,
    stratify=train_val["patient_cancer"],
)

remaining_2, triage_train = train_test_split(
    remaining_1,
    test_size=round(15 / 90, 6),  # ≈ 0.1667
    random_state=SEED,
    stratify=remaining_1["patient_cancer"],
)

cnn_train, classifier_train = train_test_split(
    remaining_2,
    test_size=round(15 / 75, 6),  # = 0.20
    random_state=SEED,
    stratify=remaining_2["patient_cancer"],
)

cnn_train.reset_index(drop=True, inplace=True)
classifier_train.reset_index(drop=True, inplace=True)
triage_train.reset_index(drop=True, inplace=True)
val.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)

print(f"Number of patients in CNN training set: {len(cnn_train)}")
print(f"Number of patients in Classifier training set: {len(classifier_train)}")
print(f"Number of patients in Triage training set: {len(triage_train)}")
print(f"Number of patients in Validation set: {len(val)}")
print(f"Number of patients in Test set: {len(test)}")

Number of patients in CNN training set: 2639
Number of patients in Classifier training set: 660
Number of patients in Triage training set: 661
Number of patients in Validation set: 440
Number of patients in Test set: 600


#### Map the patient level splits back to the breast level dataframe

In [17]:
cnn_patients = cnn_train["study_id"].tolist()
classifier_patients = classifier_train["study_id"].tolist()
triage_patients = triage_train["study_id"].tolist()
val_patients = val["study_id"].tolist()
test_patients = test["study_id"].tolist()

# Map the patient level labels back to the original metadata
cnn_df = metadata_df[metadata_df["study_id"].isin(cnn_patients)].reset_index(drop=True)
classifier_df = metadata_df[metadata_df["study_id"].isin(classifier_patients)].reset_index(drop=True)
triage_df = metadata_df[metadata_df["study_id"].isin(triage_patients)].reset_index(drop=True)
val_df = metadata_df[metadata_df["study_id"].isin(val_patients)].reset_index(drop=True)
test_df = metadata_df[metadata_df["study_id"].isin(test_patients)].reset_index(drop=True)

# Merge all training dataframes
train_df = pd.concat([cnn_df, classifier_df, triage_df], ignore_index=True)

# Standardize the age column
scaler = StandardScaler()
scaler.fit(train_df[["Patient's Age"]])
cnn_df["Patient's Age"] = scaler.transform(cnn_df[["Patient's Age"]])
classifier_df["Patient's Age"] = scaler.transform(classifier_df[["Patient's Age"]])
triage_df["Patient's Age"] = scaler.transform(triage_df[["Patient's Age"]])
val_df["Patient's Age"] = scaler.transform(val_df[["Patient's Age"]])
test_df["Patient's Age"] = scaler.transform(test_df[["Patient's Age"]])

# Save the age scaler
age_encoder_path = os.path.join("models", "age_encoder.pkl")

with open(age_encoder_path, "wb") as f:
    pickle.dump(scaler, f)

cnn_df.rename(columns={"Patient's Age": "age"}, inplace=True)
classifier_df.rename(columns={"Patient's Age": "age"}, inplace=True)
triage_df.rename(columns={"Patient's Age": "age"}, inplace=True)
val_df.rename(columns={"Patient's Age": "age"}, inplace=True)
test_df.rename(columns={"Patient's Age": "age"}, inplace=True)

In [18]:
class_counts = cnn_df["diagnosis"].value_counts().tolist()
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = class_weights[cnn_df["diagnosis"].tolist()].tolist()

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

In [19]:
cnn_train_dataset = MammoCNNDataset(cnn_df, is_training=True)
val_dataset = MammoCNNDataset(val_df, is_training=False)
test_dataset = MammoCNNDataset(test_df, is_training=False)

cnn_train_loader = DataLoader(
    cnn_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    sampler=weighted_sampler,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
)

# Examine the train dataset
patient_batch = next(iter(cnn_train_loader))

print(f"Image Shape: {patient_batch['mammogram'].shape}")
print(f"Diagnosis Shape: {patient_batch['diagnosis'].shape}")
print(f"Sign Shape: {patient_batch['sign'].shape}")
print(f"Suspicion Shape: {patient_batch['suspicion'].shape}")
print(f"Density Shape: {patient_batch['density'].shape}")
print(f"Age Shape: {patient_batch['age'].shape}")

Image Shape: torch.Size([32, 3, 320, 416])
Diagnosis Shape: torch.Size([32])
Sign Shape: torch.Size([32, 10])
Suspicion Shape: torch.Size([32, 5])
Density Shape: torch.Size([32, 4])
Age Shape: torch.Size([32])
